# Rețele neuronale și deep learning — Seminarul 4: Rețele neuronale convoluționale (CNN) și transferul învățării
Prof. dr. Daniel Traian Pele — Academia de Studii Economice din București (ASE); Academia Română, Institutul de Prognoză Economică (IPE)

[Deschide în Colab](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/main/notebooks/L4_seminar_ro.ipynb)

Sarcinile seminarului, pe datele reale din `../data/clean` (vezi `../data/SOURCES.md`). Scrieți codul în celulele goale de sub fiecare sarcină; păstrați valoarea fixă a generatorului aleator, ca rezultatele să poată fi comparate.

In [ ]:
# Google Colab: copiem repo-ul cursului, ca datele și rezultatele salvate să fie găsite
import os, subprocess, sys
if 'google.colab' in sys.modules and not os.getcwd().startswith('/content/NEURAL_BIZ'):
    if not os.path.isdir('/content/NEURAL_BIZ'):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/danpele/NEURAL_BIZ.git', '/content/NEURAL_BIZ'], check=True)
    os.chdir('/content/NEURAL_BIZ/notebooks')

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
DATA = '../data/clean/'
SEED = 20261108
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
except ImportError:
    DEVICE = 'cpu'
plt.rcParams.update({'figure.facecolor': 'none', 'axes.facecolor': 'none', 'axes.grid': False,
                     'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False})
print('device:', DEVICE)

## Pregătire

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
VISION = '../data/raw/vision'                 # torchvision downloads Fashion-MNIST and EuroSAT here on first use
os.makedirs(VISION, exist_ok=True)
fm_tr = torchvision.datasets.FashionMNIST(VISION, train=True, download=True)
fm_te = torchvision.datasets.FashionMNIST(VISION, train=False, download=True)
X = fm_tr.data.numpy().astype(np.float32) / 255; y = fm_tr.targets.numpy()
Xte = fm_te.data.numpy().astype(np.float32) / 255; yte = fm_te.targets.numpy()
perm = np.random.RandomState(SEED).permutation(len(X)); itr, iva = perm[:50000], perm[50000:]
MU, SD = X[itr].mean(), X[itr].std()            # scaling fitted on the training part only
T = lambda a: torch.tensor((a - MU) / SD).unsqueeze(1)
CLASSES = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
print(len(itr), len(iva), len(Xte), round(float(MU), 4), round(float(SD), 4))

In [ ]:
def evaluate(m, Xt, yt, bs=1000):
    m.eval(); out = []
    with torch.no_grad():
        for j in range(0, len(Xt), bs):
            out.append(m(Xt[j:j + bs].to(DEVICE)).cpu())
    p = torch.cat(out).argmax(1).numpy()
    return float((p == yt).mean()), p

def fit(m, Xtr, ytr, Xva, yva, epochs=6, lr=1e-3, bs=128):
    torch.manual_seed(SEED); m = m.to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3 * lr, total_steps=epochs * int(np.ceil(len(Xtr) / bs)))
    yt = torch.tensor(ytr); best, state = -1, None
    for ep in range(epochs):
        m.train(); idx = torch.randperm(len(Xtr))
        for j in range(0, len(idx), bs):
            b = idx[j:j + bs]
            loss = F.cross_entropy(m(Xtr[b].to(DEVICE)), yt[b].to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step(); sch.step()
        va, _ = evaluate(m, Xva, yva)
        if va > best: best, state = va, {k: v.detach().clone() for k, v in m.state_dict().items()}
        print(f'epoch {ep + 1}: validation accuracy {va:.4f}')
    m.load_state_dict(state)                  # early stopping: best validation epoch
    return m

def cnn():
    blk = lambda a, b: [nn.Conv2d(a, b, 3, padding=1), nn.BatchNorm2d(b), nn.ReLU()]
    return nn.Sequential(*blk(1, 32), *blk(32, 32), nn.MaxPool2d(2), *blk(32, 64), *blk(64, 64), nn.MaxPool2d(2),
                         *blk(64, 128), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, 10))
nparams = lambda m: sum(p.numel() for p in m.parameters())

**Sarcina 1.** Calculați de mînă dimensiunea ieșirii și numărul de parametri ai unei convoluții cu $k=5$, $p=2$, $s=2$, 3 canale de intrare și 16 filtre, pe o imagine de $64\times64$; verificați cu `nn.Conv2d`.

In [ ]:
# codul vostru aici

**Sarcina 2.** Antrenați CNN-ul din curs cu jumătate din filtre în fiecare strat (16, 16, 32, 32, 64), timp de 4 epoci; comparați acuratețea pe test și numărul de parametri cu CNN-ul complet.

In [ ]:
# codul vostru aici

**Sarcina 3.** Aplicați testul McNemar între cele două rețele din sarcina 2, pe cele 10.000 de imagini de test. Este diferența semnificativă la 5%?

In [ ]:
# codul vostru aici

## EuroSAT

In [ ]:
from PIL import Image
es = torchvision.datasets.EuroSAT(VISION, download=True)            # 27,000 Sentinel-2 tiles, 64 x 64 RGB
root = os.path.join(VISION, 'eurosat', '2750')
DIRS = sorted(os.listdir(root)); DIRS = [d for d in DIRS if os.path.isdir(os.path.join(root, d))]
XE, YE = [], []
for c, d in enumerate(DIRS):
    for f in sorted(os.listdir(os.path.join(root, d))):
        if f.endswith('.jpg'):
            XE.append(np.asarray(Image.open(os.path.join(root, d, f)).convert('RGB'))); YE.append(c)
XE, YE = np.stack(XE), np.array(YE)
rs = np.random.RandomState(SEED); etr, eva, ete = [], [], []
for c in range(10):
    ids = rs.permutation(np.where(YE == c)[0]); n = len(ids)
    etr += list(ids[:int(.6 * n)]); eva += list(ids[int(.6 * n):int(.8 * n)]); ete += list(ids[int(.8 * n):])
etr, eva, ete = map(np.array, (etr, eva, ete))
MUI = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1); SDI = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
def es_t(ids, size=112):
    x = (torch.tensor(XE[ids]).permute(0, 3, 1, 2).float() / 255 - MUI) / SDI
    return F.interpolate(x, size=(size, size), mode='bilinear', align_corners=False)
print(DIRS, len(etr), len(eva), len(ete))

**Sarcina 4.** Extrageți trăsăturile înghețate ResNet-18 (512 pentru fiecare imagine) și estimați o regresie logistică cu 25 de imagini etichetate pe clasă. Raportați acuratețea pe test și perechea de clase cel mai des confundată.

In [ ]:
# codul vostru aici

**Sarcina 5.** Întrebare de afaceri: cu modelul din sarcina 4, estimați ponderea imaginilor 'AnnualCrop' + 'PermanentCrop' în setul de test și comparați-o cu ponderea reală.

In [ ]:
# codul vostru aici

**Sarcina 6.** Repetați sarcina 4 cu 10, 50 și 250 de imagini pe clasă și reprezentați grafic acuratețea în funcție de bugetul de etichete (scară logaritmică). Unde v-ați opri cu etichetarea?

In [ ]:
# codul vostru aici